# MoReV2X guide: start here

This repository is a fork of [LLusvarghi/MoReV2X](https://github.com/LLusvarghi/MoReV2X), a simulator of NR-V2X sidelink Mode 2.

**This branch runs on ns-3.42**, the CTTC v2x tree of your groupcast work. `develop` stays on ns-3.22. Notebooks 02 to 05 keep their measurements from ns-3.22, which the port reproduces: see [notebook 06](06_ns3_migration.ipynb).

| Notebook | Question it answers |
|---|---|
| [01 Fork comparison](01_fork_comparison.ipynb) | What do the 8 forks contain, and what should be merged? |
| [02 Mode 2 resource allocation](02_mode2_resource_allocation.ipynb) | How does the scheduler work, and how do I tune it? |
| [03 Per-node schedulers](03_per_node_schedulers.ipynb) | Can nodes run different schedulers? |
| [04 SUMO mobility](04_sumo_mobility.ipynb) | Is SUMO supported? How is it mounted? |
| [05 Metrics](05_metrics.ipynb) | Which metrics exist, and how are they collected? |
| [06 Migration to ns-3.42](06_ns3_migration.ipynb) | What the port took, and how it was validated |
| [07 Mode 2 in 5G-LENA and MoReV2X](07_mode2_v2x_and_morev2x.ipynb) | How each implements it, and how they differ |

Every command, for days without AI: [`COMMANDS.txt`](../../COMMANDS.txt).

## Open the environment

```bash
source .env
```

It sets Python 3 and SUMO 1.28 from `.venv`, and the compiler flags, in this shell only. `morev2x_env_off` undoes it. Nothing is installed globally and nothing in `groupcast-adaptation-ns3` is written to. Your usual `SUMO_HOME` points inside that simulator, so `.env` replaces it for the shell.

## What differs from upstream

Upstream does not run on macOS: it fails to compile, and after fixing that it runs without V2X tags. Everything below was needed to get a working simulator, or was asked for. On this branch the ns-3 tree itself is also replaced: the `VERSION` and reference file items below applied to ns-3.22.

### Needed for macOS
| Change | Why |
|---|---|
| `VERSION` renamed to `VERSION.txt` | On a case-insensitive disk it shadows the C++ header `<version>`, giving 23,818 compile errors |
| `UdpClient::Send` and `PacketSink::HandleRead` marked `weak` | `scratch/HIGHWAY.cc` redefines both. Without `weak`, macOS keeps the library versions, packets lose their `NrV2XTag`, PDB is uninitialised memory, and the selection window grows until the machine runs out of memory |
| `-ffp-contract=off` in `.env` | arm64 fuses multiply-adds, which shifts floating-point results by 1 ulp compared with x86 |
| `int64x64` test accepts a 64-bit `long double` | Extends the existing Valgrind branch (ns-3 bug 1882) |
| 81 ns-3 reference files restored | Upstream dropped the `.pcap`, `.mob` and `.dat` test vectors. Restored from the official `ns-3.22` tag |

### Behaviour of the simulator
| Change | Default | Why |
|---|---|---|
| Empty candidate list falls back to the full window | always | With a short RRI, own past transmissions can exclude everything. Upstream then indexed an empty vector, which is undefined behaviour in an optimized build. Its authors had left this same fallback commented out |
| RRI 20 ms added to the list in mixed traffic | always | Nodes used a 20 ms RRI that was not in the list, so `--Mixed` failed an assertion in debug builds |
| `LegacyEvenUeOneShot` MAC attribute | **off** | Upstream forced `Cresel=1` on every even RNTI, a leftover from a paper review. Half the fleet silently never used SPS. **This changes results compared with upstream v4.1.** `--LegacyEvenUeOneShot=true` restores it |
| `MetricsMinX/MaxX`, `CbrMinX/MaxX` attributes | upstream values | The logs only covered a hardcoded 0 to 3500 m highway |

### Merged from forks
| Change | From |
|---|---|
| `Pkeep` attribute, `--pkeep`, and removal of the `std::cin.get()` that stopped such runs | kaefcatcher |
| `SubframeInfo::operator-` wrap fixed | vivatttt |
| MAC members initialised in the constructor | vivatttt |
| `tools/run_sweep.py`, resumable YAML sweeps | ported from kaefcatcher's `run_simulator.py` |
| Adaptive reservation, `--AdaptiveRRI` and `adaptive` in `--SchedulerMap`, off by default | rewritten from vivatttt, see [notebook 01](01_fork_comparison.ipynb) |

`tools/fingerprint.sh` hashes the sorted Mode 2 logs of 3 fixed runs. It stayed identical through all of these merges, so default behaviour did not change.

### New
- `src/traci`: TraCI client, ported from the groupcast simulator, and `SumoMobilityHelper`.
- `HIGHWAY` options: `--sumoCfg`, `--mobilityTrace`, `--sumoWarmup`, `--SchedulerMap`, `--pkeep`, `--PosLog`, `--UdpFiles`.
- Tests: suites `nr-v2x-mode2` and `sumo-mobility`, and `tests/test_integration.py`.
- `tools/gate.sh` and the git hooks that block a commit until it passes.

### Removed
`src/MoReV2X/examples/HIGHWAY.cc`, an old copy of the scenario that aborted at startup because it sets an attribute that no longer exists (`AggressiveMode4`). `scratch/HIGHWAY.cc` is the real one.

## Test status

`tools/gate.sh` runs the build, every ns-3 suite and example, and the scenario tests. Its logs are in `build/gate/`. `tools/debug-check.sh` runs all 22 scheduler modes with `NS_ASSERT` active, which the optimized build ignores. Run it after changing the MAC.

In [1]:
import os, subprocess, sys
sys.path.insert(0, os.path.abspath("../../tools"))
import morev2x_results as mr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 90)
mr.apply_chart_style()
os.chdir(mr.ROOT)

In [2]:
log = os.path.join(mr.ROOT, "build", "gate", "ns3-tests.log")
if os.path.exists(log):
    lines = open(log).read().splitlines()
    print(next(l for l in lines if "tests passed" in l))
    print("\n".join(l for l in lines if l.startswith(("FAIL", "CRASH"))))
    print("\n".join(l for l in lines if l.startswith("PASS") and ("nr-v2x-mode2" in l or "sumo-mobility" in l)))
print(open(os.path.join(mr.ROOT, "tools", "known-failures.txt")).read())

317 of 328 tests passed (317 passed, 10 skipped, 1 failed, 0 crashed, 0 valgrind errors)
FAIL: TestSuite lte-x2-handover
PASS: TestSuite nr-v2x-mode2
PASS: TestSuite sumo-mobility
# ns-3 suites allowed to fail in tools/gate.sh. One suite name per line. None on ns-3.42.



On ns-3.42 every suite passes. On `develop`, `lte-x2-handover` fails on ns-3.22, and it is out of scope there.

## A first run

Each run writes one folder under `results/`. This one takes a few seconds.

In [3]:
run = mr.simulate("start", "--Vehicles=20", "--simTime=5", "--Periodic", "--mcs=13", "--seed=1")
print(run.folder)
print(sorted(os.listdir(run.folder)))
mr.summary(run).round(3)

/Users/hadih/Documents/MoReV2X-hadih/results/notebooks/start/results/Periodic_Dynamic0_avgRRI0_VariableSize0_ReEval0_20_PDB0_1_1
['CBR_OutputFile.txt', 'ReEvaluationsLog.txt', 'ReceivedLog.txt', 'ReservationsLog.txt', 'SSPSlog.txt', 'SchedulerMap.txt', 'phyDebugShort.txt', 'simREADME.txt']


PDR near               0.973
collision share        0.015
CBR                    0.050
latency p95 [ms]      92.900
selections            82.000
transmissions       1260.000
dtype: float64